In [1]:
import requests

In [2]:
normal = requests.get('http://quotes.toscrape.com/').text
js_rendered = requests.get('http://quotes.toscrape.com/js/').text

In [3]:
print('Normal page length: ', len(normal))
print('JS page length: ', len(js_rendered))

Normal page length:  11021
JS page length:  5806


In [4]:
print('Albert Einstein' in normal)
print('Albert Einstein' in js_rendered)

True
True


In [5]:
# find where "Albert Einstein" appears in the JS-rendered page
idx = js_rendered.find('Albert Einstein')
print(js_rendered[idx-300:idx+100])


                </p>
            </div>
        </div>
    
<script src="/static/jquery.js"></script>
<script>
    var data = [
    {
        "tags": [
            "change",
            "deep-thoughts",
            "thinking",
            "world"
        ],
        "author": {
            "name": "Albert Einstein",
            "goodreads_link": "/author/show/9810.Albert_Einstein",
            "sl


### We can use trick here, because it happened that all data dumped as JS variables right in the page source, no need Playwright here

In [6]:
from bs4 import BeautifulSoup

soup_normal = BeautifulSoup(normal, 'html.parser')
soup_js = BeautifulSoup(js_rendered, 'html.parser')

print("Normal page - quote spans found:", len(soup_normal.find_all('span', class_='text')))
print("JS page - quote spans found:", len(soup_js.find_all('span', class_='text')))

Normal page - quote spans found: 10
JS page - quote spans found: 0


In [7]:
response = requests.get('http://quotes.toscrape.com/js/')
soup = BeautifulSoup(response.text, 'html.parser')

scripts = soup.find_all('script')
print(f'Number of script tags there is {len(scripts)}')

Number of script tags there is 2


In [10]:
for i, s in enumerate(scripts):
    preview = (s.string or '')[:60]
    print(f'{i}->{preview}')

0->
1->
    var data = [
    {
        "tags": [
            "chang


In [11]:
target_script = None
for s in scripts:
    if s.string and 'var data' in s.string:
        target_script = s.string
        break

print(target_script[:300])


    var data = [
    {
        "tags": [
            "change",
            "deep-thoughts",
            "thinking",
            "world"
        ],
        "author": {
            "name": "Albert Einstein",
            "goodreads_link": "/author/show/9810.Albert_Einstein",
            "slug": "Alber


## Regex quick reference — extracting embedded JSON from a `<script>` tag

**Goal:** pull `[ ... ]` out of `var data = [ ... ];` so it can be parsed with `json.loads()`.

```python
import re
match = re.search(r'var data = (\[.*?\]);', script_text, re.DOTALL)
json_text = match.group(1)
```

**Pattern breakdown — `r'var data = (\[.*?\]);'`**

| Piece | Meaning |
|---|---|
| `var data = ` | matches this literal text exactly |
| `\[` | matches a literal `[` (escaped, since `[` is a special regex character) |
| `.*?` | matches "any characters" — the `?` makes it **non-greedy**, i.e. stop at the *first* `]`, not the last one in the whole file |
| `\]` | matches a literal `]` |
| `( ... )` | a **capturing group** — marks the part we actually want to extract |
| `;` | matches the trailing semicolon |

**Why `re.DOTALL`:** by default `.` does NOT match newlines. Since the JSON spans multiple lines, `DOTALL` makes `.` match everything, including `\n`.

**Why non-greedy (`*?`) matters:** without it, `.*` grabs as much as possible — it would match from the first `[` all the way to the *last* `]` anywhere later in the script, potentially swallowing unrelated code. `*?` stops at the nearest valid closing `]`.

**`match.group(1)`:** returns just what's inside the parentheses (the array text) — not the whole matched string (which would include `var data = ` and `;` too).

**Next step:** `json.loads(json_text)` turns that string into a real Python `list` of `dict`s.

In [13]:
import re

match = re.search(r'var data = (\[.*?\]);', target_script, re.DOTALL)
json_text = match.group(1)
print(json_text[:200])

[
    {
        "tags": [
            "change",
            "deep-thoughts",
            "thinking",
            "world"
        ],
        "author": {
            "name": "Albert Einstein",
         


In [14]:
import json

data = json.loads(json_text)

print(type(data))
print(len(data))
print(data[0])

<class 'list'>
10
{'tags': ['change', 'deep-thoughts', 'thinking', 'world'], 'author': {'name': 'Albert Einstein', 'goodreads_link': '/author/show/9810.Albert_Einstein', 'slug': 'Albert-Einstein'}, 'text': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'}


### Working with Playwright

In [15]:
%pip install playwright

   ---------------------------------------- 0.0/38.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.2 MB ? eta -:--:--
    --------------------------------------- 0.5/38.2 MB 2.1 MB/s eta 0:00:18
   - -------------------------------------- 1.3/38.2 MB 2.9 MB/s eta 0:00:13
   - -------------------------------------- 1.8/38.2 MB 3.0 MB/s eta 0:00:13
   --- ------------------------------------ 2.9/38.2 MB 3.4 MB/s eta 0:00:11
   ---- ----------------------------------- 4.2/38.2 MB 3.9 MB/s eta 0:00:09
   ----- ---------------------------------- 5.5/38.2 MB 4.3 MB/s eta 0:00:08
   ------- -------------------------------- 7.1/38.2 MB 4.7 MB/s eta 0:00:07
   -------- ------------------------------- 8.4/38.2 MB 5.0 MB/s eta 0:00:06
   ---------- ----------------------------- 9.7/38.2 MB 5.3 MB/s eta 0:00:06
   ------------ --------------------------- 11.5/38.2 MB 5.5 MB/s eta 0:00:05
   ------------- --

In [16]:
!playwright install chromium

|                                                                                |   0% of 191.8 MiB
|■■■■■■■■                                                                        |  10% of 191.8 MiB
|■■■■■■■■■■■■■■■■                                                                |  20% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■                                                        |  30% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                                |  40% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                        |  50% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                |  60% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                        |  70% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                |  80% of 191.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■        |  90% of 

In [17]:
from playwright.sync_api import sync_playwright

In [ ]:
with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto('http://quotes.toscrape.com/js/')

    html = page.content()
    print(len(html))

    browser.close()

##### This gave an error because Playwright was trying to create second loop while there is already jupyter loop here

In [19]:
from playwright.async_api import async_playwright

In [26]:
async def get_rendered_html(url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url)
        html = await page.content()
        await browser.close()
        return html

In [ ]:
html = await get_rendered_html('http://quotes.toscrape.com/js/')
print(len(html))

#### again error!!! lets try without event loop of its own at all